In [ ]:
"""
Stage 4 (PD vs HC) - Multi-classifier nested CV at five panel sizes
====================================================================

For each panel size in {5, 7, 10, 12, 16} genes (top-K by fold_inclusion
in consensus_panel.csv from the PD vs HC Stage-3 no-SMOTE run):

  Phase 0  Subset train/test to top-K consensus genes (saved as
           train_postfeature.csv / test_postfeature.csv inside the
           panel-specific output folder).

  Phase 1  Nested CV on the train half:
              outer = 9 stratified folds x 3 seeds (42, 123, 256) = 27 folds
              inner = 4 stratified folds, GridSearchCV with RobustScaler
                      INSIDE the pipeline (refit per inner fold).
           Inner-CV scoring metric: balanced_accuracy.
           Sample weighting: composite (sex x cell_type) stratum balance
           multiplied by class balance (HC vs PD inverse frequency),
           applied to RBF, Linear, GaussianNB, ElasticNet (the classifiers
           whose fit() supports sample_weight).

Class encoding: NEG=HC=0 (minority, ~37 train), POS=PD=1 (majority, ~71 train).

Inputs:
    train.csv      Outputs/stage2_split_stratum_aware_PDHC/train.csv
    test.csv       Outputs/stage2_split_stratum_aware_PDHC/test.csv
    metadata       Meta_data_PDHC.csv   (filtered to train / test sample IDs)
    consensus      Outputs/stage3_feature_selection_PDHC/consensus_panel.csv

Outputs per panel size (in PDHC new/Outputs/stage4_classifiers_PDHC_topK/):
    train_postfeature.csv, test_postfeature.csv
    fold_results.csv               per-(seed,fold,classifier) HPs + metrics
    consensus_hyperparams.csv      mode HP per classifier across 27 folds
    final_metrics.csv              mean +- std for full metric battery
    roc_curves.png                 binary ROC per classifier
    pr_curves.png                  binary PR per classifier
    enet_fold_inclusion.csv        gene -> % of folds non-zero in ENet
    rfe_fold_inclusion.csv         gene -> % of folds in RFE support
"""

from __future__ import annotations

import json
import warnings
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    cohen_kappa_score, confusion_matrix, f1_score, matthews_corrcoef,
    precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

# ----------------------------- Configuration --------------------------------
PIPELINE_DIR  = Path(r"C:\Users\hafsa\Python PD Project\MI_BaggedLASSO Pipeline")
PDHC_DIR      = PIPELINE_DIR / "PDHC new"
SPLIT_DIR     = PDHC_DIR / "Outputs" / "stage2_split_stratum_aware_PDHC"
STAGE3_DIR    = PDHC_DIR / "Outputs" / "stage3_feature_selection_PDHC"
STAGE4_BASE   = PDHC_DIR / "Outputs" 

CONSENSUS_PATH = STAGE3_DIR / "consensus_panel.csv"
TRAIN_PATH     = SPLIT_DIR / "train.csv"
TEST_PATH      = SPLIT_DIR / "test.csv"
META_PATH      = PDHC_DIR / "Meta_data_PDHC.csv"   # full metadata; filtered by sample IDs

CONDITION_COL = "condition"
PANEL_SIZES   = [5, 7, 10, 16]
SEEDS         = [42, 123, 256]
N_OUTER       = 9
N_INNER       = 4
N_JOBS        = -1

# Class encoding: HC = 0 (negative, minority), PD = 1 (positive, majority)
NEG_LABEL = "HC"
POS_LABEL = "PD"


# ============================ Data helpers =================================
def load_meta(sample_ids) -> pd.DataFrame:
    """Read full Meta_data_PDHC.csv and align rows to sample_ids."""
    meta = pd.read_csv(META_PATH)
    if "sample_id" in meta.columns:
        meta = meta.set_index("sample_id")
    return meta.loc[sample_ids]


def to_binary_labels(y_str: np.ndarray) -> np.ndarray:
    """HC stays HC; everything else collapsed to PD (data is already binary)."""
    return np.where(np.asarray(y_str) == NEG_LABEL, NEG_LABEL, POS_LABEL)


def encode_binary(y_str_binary: np.ndarray) -> np.ndarray:
    """HC -> 0, PD -> 1."""
    return np.where(y_str_binary == POS_LABEL, 1, 0).astype(int)


def load_features_x_samples(path: Path, top_genes: List[str]) -> pd.DataFrame:
    """Load CSV; ensure genes are the row index; subset to top_genes (in order)."""
    df = pd.read_csv(path, index_col=0)
    if len(set(top_genes) & set(df.index.astype(str))) == 0:
        df = df.T
    present = [g for g in top_genes if g in df.index]
    missing = [g for g in top_genes if g not in df.index]
    if missing:
        print(f"  WARN: {len(missing)} genes missing in {path.name}")
    return df.loc[present]


def composite_sample_weights(meta: pd.DataFrame,
                             y_int: np.ndarray) -> np.ndarray:
    """Composite weight = confounder (sex x cell_type) stratum balance
    multiplied by class balance (HC vs PD inverse frequency), normalised
    to mean 1. The class term mirrors sklearn's class_weight='balanced':
    w_c = n / (n_classes * count_c)."""
    strata = meta["sex"].astype(str) + "_" + meta["cell_type"].astype(str)
    s_counts = strata.value_counts()
    n, k = len(strata), len(s_counts)
    w_conf = strata.map(lambda s: (n / k) / s_counts[s]).values.astype(float)

    classes, c_counts = np.unique(y_int, return_counts=True)
    n_cls = len(classes)
    class_w = {c: n / (n_cls * cnt) for c, cnt in zip(classes, c_counts)}
    w_class = np.array([class_w[y] for y in y_int], dtype=float)

    w = w_conf * w_class
    return w / w.mean()


# ============================ Classifiers ==================================
def get_classifiers(seed: int) -> Dict[str, Dict]:
    return {
        "mSVM-RBF": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", SVC(kernel="rbf", probability=True,
                                          random_state=seed))]),
            "grid": {"clf__C":     [0.1, 1, 10],
                     "clf__gamma": ["scale", 0.01, 0.1, 1.0]},
            "supports_sw": True,
        },
        "mSVM-Linear": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", SVC(kernel="linear", probability=True,
                                          random_state=seed))]),
            "grid": {"clf__C": [0.01, 0.1, 1, 10]},
            "supports_sw": True,
        },
        "mSVM-RFE": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("rfe", RFE(SVC(kernel="linear", probability=True,
                                              random_state=seed)))]),
            "grid": {"rfe__n_features_to_select": None,    # filled per panel
                     "rfe__estimator__C":         [0.1, 1, 10]},
            "supports_sw": False,
        },
        "kNN": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", KNeighborsClassifier())]),
            "grid": {"clf__n_neighbors": [3, 5, 7, 11, 15],
                     "clf__weights":     ["uniform", "distance"]},
            "supports_sw": False,
        },
        "GaussianNB": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", GaussianNB())]),
            "grid": {"clf__var_smoothing": np.logspace(-11, -7, 5)},
            "supports_sw": True,
        },
        "ElasticNet": {
            "pipe": Pipeline([("scale", RobustScaler()),
                              ("clf", LogisticRegression(
                                  penalty="elasticnet", solver="saga",
                                  max_iter=5000, tol=1e-3,
                                  random_state=seed))]),
            "grid": {"clf__C":        [0.01, 0.1, 1, 10],
                     "clf__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]},
            "supports_sw": True,
        },
    }


def rfe_grid_for_panel(k_top: int) -> List[int]:
    """Sensible n_features_to_select grid given the panel size."""
    base = sorted({max(2, k_top // 4), max(3, k_top // 2), k_top - 1, k_top})
    return [b for b in base if 2 <= b <= k_top]


# ============================ Metrics ======================================
def evaluate_binary(y_true: np.ndarray, y_pred: np.ndarray,
                    y_proba_pos: np.ndarray) -> Dict:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1":          f1_score(y_true, y_pred, average="macro"),
        "auroc":             roc_auc_score(y_true, y_proba_pos),
        "pr_auc":            average_precision_score(y_true, y_proba_pos),
        "sensitivity":       sens,
        "specificity":       spec,
        "mcc":               matthews_corrcoef(y_true, y_pred),
        "brier":             brier_score_loss(y_true, y_proba_pos),
        "cohen_kappa":       cohen_kappa_score(y_true, y_pred),
    }


def selected_genes_from_estimator(name: str, gs_result, gene_names: List[str]):
    best = gs_result.best_estimator_
    if name == "ElasticNet":
        coef = best.named_steps["clf"].coef_
        mask = (np.abs(coef) > 1e-10).any(axis=0)
        return [g for g, m in zip(gene_names, mask) if m]
    if name == "mSVM-RFE":
        rfe = best.named_steps["rfe"]
        return [g for g, m in zip(gene_names, rfe.support_) if m]
    return []


# ============================ Aggregation helpers ===========================
def aggregate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["balanced_accuracy", "macro_f1", "auroc", "pr_auc",
                   "sensitivity", "specificity", "mcc", "brier", "cohen_kappa"]
    rows = []
    for clf, sub in df.groupby("classifier"):
        row = {"classifier": clf, "n_folds": len(sub)}
        for m in metric_cols:
            row[f"{m}_mean"] = sub[m].mean()
            row[f"{m}_std"]  = sub[m].std()
        rows.append(row)
    return pd.DataFrame(rows)


def consensus_hyperparams(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for clf, sub in df.groupby("classifier"):
        params_list = [json.loads(p) for p in sub["best_params"]]
        keys = sorted({k for p in params_list for k in p.keys()})
        row = {"classifier": clf}
        for k in keys:
            vals = [p.get(k) for p in params_list]
            counter = Counter([str(v) for v in vals])
            mode_str, mode_count = counter.most_common(1)[0]
            row[k] = mode_str
            row[f"{k}__support_pct"] = round(100 * mode_count / len(vals), 1)
        rows.append(row)
    return pd.DataFrame(rows)


def fold_inclusion_table(sel_records, top_genes, n_folds_total):
    df = pd.DataFrame(sel_records)
    out = {}
    if df.empty:
        return out
    for clf in df["classifier"].unique():
        sub = df[df["classifier"] == clf]
        n_seen = sub.groupby("gene").size()
        full = pd.Series(0, index=top_genes, dtype=int)
        full.update(n_seen)
        out[clf] = pd.DataFrame({
            "gene": full.index,
            "n_folds_included": full.values,
            "fold_inclusion_pct":
                np.round(100 * full.values / n_folds_total, 1),
        }).sort_values("n_folds_included", ascending=False)
    return out


# ============================ Plot helpers =================================
def plot_roc_per_classifier(fold_df: pd.DataFrame, out_path: Path):
    fig, ax = plt.subplots(figsize=(8, 6))
    for clf, sub in fold_df.groupby("classifier"):
        y_true_all = np.concatenate([np.asarray(t) for t in sub["y_true"]])
        y_proba_all = np.concatenate([np.asarray(p) for p in sub["y_proba_pos"]])
        fpr, tpr, _ = roc_curve(y_true_all, y_proba_all)
        auc = roc_auc_score(y_true_all, y_proba_all)
        ax.plot(fpr, tpr, label=f"{clf} (AUC={auc:.3f})", linewidth=1.7)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, linewidth=1)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC curves across 27 outer folds (concatenated) - PD vs HC")
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(out_path, dpi=200); plt.close(fig)


def plot_pr_per_classifier(fold_df: pd.DataFrame, out_path: Path):
    fig, ax = plt.subplots(figsize=(8, 6))
    for clf, sub in fold_df.groupby("classifier"):
        y_true_all = np.concatenate([np.asarray(t) for t in sub["y_true"]])
        y_proba_all = np.concatenate([np.asarray(p) for p in sub["y_proba_pos"]])
        prec, rec, _ = precision_recall_curve(y_true_all, y_proba_all)
        ap = average_precision_score(y_true_all, y_proba_all)
        ax.plot(rec, prec, label=f"{clf} (AP={ap:.3f})", linewidth=1.7)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("PR curves across 27 outer folds (concatenated) - PD vs HC")
    ax.legend(loc="lower left", fontsize=9)
    ax.grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(out_path, dpi=200); plt.close(fig)


# ============================ Per-classifier CV ============================
def run_classifier_cv(name, cfg, X, y_int, gene_names, meta,
                      fold_records, sel_records):
    print(f"\n--- {name} ---")
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=seed)
        for fold_i, (tr_idx, te_idx) in enumerate(skf.split(X, y_int)):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y_int[tr_idx], y_int[te_idx]
            sw_tr = composite_sample_weights(meta.iloc[tr_idx], y_int[tr_idx])

            inner = StratifiedKFold(n_splits=N_INNER, shuffle=True,
                                    random_state=seed * 1000 + fold_i)
            gs = GridSearchCV(
                estimator=cfg["pipe"],
                param_grid=cfg["grid"],
                scoring="balanced_accuracy",
                cv=inner, n_jobs=N_JOBS, refit=True,
            )

            fit_kwargs = {}
            if cfg["supports_sw"]:
                fit_kwargs["clf__sample_weight"] = sw_tr

            gs.fit(X_tr, y_tr, **fit_kwargs)

            y_pred  = gs.predict(X_te)
            y_proba = gs.predict_proba(X_te)
            y_proba_pos = y_proba[:, 1]
            metrics = evaluate_binary(y_te, y_pred, y_proba_pos)

            fold_records.append({
                "classifier":   name,
                "seed":         seed,
                "fold":         fold_i + 1,
                "best_params":  json.dumps(gs.best_params_),
                **metrics,
                "y_true":       y_te.tolist(),
                "y_proba_pos":  y_proba_pos.tolist(),
            })
            print(f"  seed={seed} fold={fold_i+1}  "
                  f"BA={metrics['balanced_accuracy']:.3f}  "
                  f"F1={metrics['macro_f1']:.3f}  "
                  f"AUROC={metrics['auroc']:.3f}  "
                  f"PR-AUC={metrics['pr_auc']:.3f}", flush=True)

            if name in ("ElasticNet", "mSVM-RFE"):
                sel = selected_genes_from_estimator(name, gs, gene_names)
                for g in sel:
                    sel_records.append({"classifier": name, "seed": seed,
                                        "fold": fold_i + 1, "gene": g})


# ============================ Per-panel run ================================
def run_for_panel(k_top: int):
    out_dir = STAGE4_BASE / f"stage4_classifiers_PDHC_top{k_top}"
    out_dir.mkdir(parents=True, exist_ok=True)
    print("\n" + "=" * 72)
    print(f"PANEL SIZE: top {k_top} consensus genes - output: {out_dir.name}")
    print("=" * 72)

    # ---- Top-K consensus genes ----
    consensus = pd.read_csv(CONSENSUS_PATH, index_col=0)
    if "fold_inclusion" not in consensus.columns:
        consensus.columns = ["fold_inclusion"]
    top_genes = (consensus.sort_values("fold_inclusion", ascending=False)
                          .head(k_top).index.tolist())
    print(f"Top genes loaded: {len(top_genes)}")

    train_df = load_features_x_samples(TRAIN_PATH, top_genes).round(4)
    test_df  = load_features_x_samples(TEST_PATH,  top_genes).round(4)
    train_df.to_csv(out_dir / "train_postfeature.csv")
    test_df.to_csv(out_dir / "test_postfeature.csv")
    print(f"Saved postfeature CSVs (train: {train_df.shape}, "
          f"test: {test_df.shape})")

    meta_train = load_meta(train_df.columns)
    y_train_str = meta_train[CONDITION_COL].values
    y_train_bin = to_binary_labels(y_train_str)
    y_train_int = encode_binary(y_train_bin)
    print(f"Class balance (PD=1, HC=0): "
          f"{dict(Counter(y_train_bin))}")

    X_train = train_df.T.values.astype(np.float64)
    gene_names = train_df.index.tolist()

    # ---- Build classifiers + RFE grid for this panel ----
    classifiers = get_classifiers(seed=42)
    classifiers["mSVM-RFE"]["grid"]["rfe__n_features_to_select"] = \
        rfe_grid_for_panel(k_top)
    print(f"RFE n_features_to_select grid: "
          f"{classifiers['mSVM-RFE']['grid']['rfe__n_features_to_select']}")

    # ---- Nested CV ----
    fold_records, sel_records = [], []
    for name, cfg in classifiers.items():
        run_classifier_cv(name, cfg, X_train, y_train_int, gene_names,
                          meta_train, fold_records, sel_records)

    fold_df = pd.DataFrame(fold_records)
    csv_cols = [c for c in fold_df.columns
                if c not in ("y_true", "y_proba_pos")]
    fold_df[csv_cols].to_csv(out_dir / "fold_results.csv", index=False)

    agg = aggregate_metrics(fold_df)
    agg.to_csv(out_dir / "final_metrics.csv", index=False)
    print("\nFinal metrics (mean across 27 folds):")
    print(agg.set_index("classifier")
          [["balanced_accuracy_mean", "macro_f1_mean",
            "auroc_mean", "pr_auc_mean"]].round(3).to_string())

    cons = consensus_hyperparams(fold_df)
    cons.to_csv(out_dir / "consensus_hyperparams.csv", index=False)
    print("\nConsensus hyperparameters (mode across 27 folds):")
    print(cons.set_index("classifier").to_string())

    plot_roc_per_classifier(fold_df, out_dir / "roc_curves.png")
    plot_pr_per_classifier(fold_df,  out_dir / "pr_curves.png")

    n_folds_total = N_OUTER * len(SEEDS)
    inc_tables = fold_inclusion_table(sel_records, top_genes, n_folds_total)
    if "ElasticNet" in inc_tables:
        inc_tables["ElasticNet"].to_csv(
            out_dir / "enet_fold_inclusion.csv", index=False)
    if "mSVM-RFE" in inc_tables:
        inc_tables["mSVM-RFE"].to_csv(
            out_dir / "rfe_fold_inclusion.csv", index=False)

    print(f"\nOutputs in: {out_dir}")


def main():
    print("=" * 72)
    print("STAGE 4 (PD vs HC) - Multi-classifier nested CV")
    print(f"Panel sizes: {PANEL_SIZES}")
    print(f"Outer={N_OUTER} x seeds={SEEDS} = {N_OUTER * len(SEEDS)} folds; "
          f"inner={N_INNER}")
    print("=" * 72)
    for k in PANEL_SIZES:
        run_for_panel(k)


if __name__ == "__main__":
    main()


STAGE 4 (PD vs HC) - Multi-classifier nested CV
Panel sizes: [5, 7, 10, 16]
Outer=9 x seeds=[42, 123, 256] = 27 folds; inner=4

PANEL SIZE: top 5 consensus genes - output: stage4_classifiers_PDHC_top5
Top genes loaded: 5
Saved postfeature CSVs (train: (5, 108), test: (5, 47))
Class balance (PD=1, HC=0): {np.str_('HC'): 36, np.str_('PD'): 72}
RFE n_features_to_select grid: [2, 3, 4, 5]

--- mSVM-RBF ---
  seed=42 fold=1  BA=0.562  F1=0.500  AUROC=0.500  PR-AUC=0.791
  seed=42 fold=2  BA=0.438  F1=0.314  AUROC=0.344  PR-AUC=0.681
  seed=42 fold=3  BA=0.625  F1=0.486  AUROC=0.469  PR-AUC=0.771
  seed=42 fold=4  BA=0.625  F1=0.486  AUROC=0.812  PR-AUC=0.896
  seed=42 fold=5  BA=0.562  F1=0.378  AUROC=0.875  PR-AUC=0.942
  seed=42 fold=6  BA=0.500  F1=0.250  AUROC=0.688  PR-AUC=0.839
  seed=42 fold=7  BA=0.625  F1=0.580  AUROC=0.594  PR-AUC=0.818
  seed=42 fold=8  BA=0.312  F1=0.294  AUROC=0.406  PR-AUC=0.724
  seed=42 fold=9  BA=0.562  F1=0.500  AUROC=0.250  PR-AUC=0.558
  seed=123 fold=1 

In [1]:
"""
Stage 4 (PD vs HC) SMOTE - Multi-panel nested CV with SMOTE inside inner folds
==========================================================================

Variant of stage4_HP_PDHC.py with SMOTE applied inside each INNER CV
fold via a custom SMOTEWeightedPipeline wrapper. Two SMOTE strategies
run sequentially against the matching Stage-3 SMOTE output (panel
sourced from the same SMOTE variant for end-to-end methodological
consistency):

  (a) BALANCED   - minority -> majority             (sampling_strategy='auto')
  (b) IMBALANCED - both classes 2x, ratio preserved (dict strategy)

Class encoding: NEG=HC=0 (minority), POS=PD=1 (majority).

LEAKAGE GUARDS (Pedregosa et al. 2011 sklearn section 3.4; Lemaitre et al.
2017 imbalanced-learn section 3):

  * SMOTE is invoked INSIDE the SMOTEWeightedPipeline wrapper, which is
    fitted by GridSearchCV per INNER training fold only. Inner validation
    folds are never touched by SMOTE.
  * Outer-test fold is held aside and never SMOTE-touched.

SAMPLE WEIGHTING:

  * Composite (sex x cell_type) x class weights are computed on
    OUTER-TRAIN real samples and passed to GridSearchCV via
    `gs.fit(..., sample_weight=sw)`.
  * GridSearchCV slices the weight vector to each inner-train fold
    automatically.
  * Inside SMOTEWeightedPipeline.fit, the inner-train weight vector is
    EXTENDED across SMOTE-synthetic samples:
        - real samples     -> retain composite weight (subset of sw)
        - synthetic samples-> uniform 1.0  (no metadata available)
      and re-normalised to mean 1.

Pipeline per inner fold (inside the wrapper):
    SMOTE -> RobustScaler -> classifier(sample_weight=ext_w)

Panel sizes (PANEL_SIZES = [5, 7, 10, 12, 16]) iterate independently;
each (panel_size x variant) produces its own output folder.

Outputs per (panel_size K, variant):
    Outputs/stage4_HP_PDHC_top{K}_smote_{variant}/
        train_postfeature.csv          original K-gene train (pre-SMOTE)
        train_postfeature_smote.csv    full-train SMOTE applied
        test_postfeature.csv           held-out test, unchanged
        fold_results.csv
        consensus_hyperparams.csv
        final_metrics.csv
        roc_curves.png, pr_curves.png
        enet_fold_inclusion.csv, rfe_fold_inclusion.csv
        smote_fold_summary.csv         pre/post SMOTE per inner fold mean
"""

from __future__ import annotations

import json
import warnings
from collections import Counter
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    cohen_kappa_score, confusion_matrix, f1_score, matthews_corrcoef,
    precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

# ----------------------------- Configuration --------------------------------
PIPELINE_DIR  = Path(r"C:\Users\hafsa\Python PD Project\MI_BaggedLASSO Pipeline")
PDHC_DIR      = PIPELINE_DIR / "PDHC new"
SPLIT_DIR     = PDHC_DIR / "Outputs" / "stage2_split_stratum_aware_PDHC"
OUTPUT_BASE   = PDHC_DIR / "Outputs"

TRAIN_PATH = SPLIT_DIR / "train.csv"
TEST_PATH  = SPLIT_DIR / "test.csv"
META_PATH  = PDHC_DIR / "Meta_data_PDHC.csv"

CONDITION_COL = "condition"
PANEL_SIZES   = [5, 7, 10, 16]
SEEDS         = [42, 123, 256]
N_OUTER       = 9
N_INNER       = 4
N_JOBS        = -1

NEG_LABEL = "HC"   # 0  (minority)
POS_LABEL = "PD"   # 1  (majority)

SMOTE_VARIANTS    = ["balanced", "imbalanced"]
SMOTE_K_NEIGHBORS = 5


# ============================ Data helpers =================================
def load_meta(sample_ids):
    meta = pd.read_csv(META_PATH)
    if "sample_id" in meta.columns:
        meta = meta.set_index("sample_id")
    return meta.loc[sample_ids]


def to_binary_labels(y_str):
    return np.where(np.asarray(y_str) == NEG_LABEL, NEG_LABEL, POS_LABEL)


def encode_binary(y_str_binary):
    return np.where(y_str_binary == POS_LABEL, 1, 0).astype(int)


def load_features_x_samples(path: Path, top_genes: List[str]) -> pd.DataFrame:
    df = pd.read_csv(path, index_col=0)
    if len(set(top_genes) & set(df.index.astype(str))) == 0:
        df = df.T
    present = [g for g in top_genes if g in df.index]
    missing = [g for g in top_genes if g not in df.index]
    if missing:
        print(f"  WARN: {len(missing)} genes missing in {path.name}")
    return df.loc[present]


def composite_sample_weights(meta: pd.DataFrame, y_int: np.ndarray) -> np.ndarray:
    """Composite weight = (sex x cell_type) stratum balance * class balance.
    Used for the REAL samples that enter outer-train. Synthetic SMOTE samples
    receive a uniform weight (1.0) inside SMOTEWeightedPipeline.fit()."""
    strata = meta["sex"].astype(str) + "_" + meta["cell_type"].astype(str)
    s_counts = strata.value_counts()
    n, k = len(strata), len(s_counts)
    w_conf = strata.map(lambda s: (n / k) / s_counts[s]).values.astype(float)

    classes, c_counts = np.unique(y_int, return_counts=True)
    n_cls = len(classes)
    class_w = {c: n / (n_cls * cnt) for c, cnt in zip(classes, c_counts)}
    w_class = np.array([class_w[y] for y in y_int], dtype=float)

    w = w_conf * w_class
    return w / w.mean()


# ============================ SMOTE strategy ================================
def smote_sampling_strategy(y_int, variant):
    if variant == "balanced":
        return "auto"
    if variant == "imbalanced":
        counts = pd.Series(y_int).value_counts().to_dict()
        return {int(c): int(2 * n) for c, n in counts.items()}
    raise ValueError(f"Unknown SMOTE variant: {variant}")


def make_smote(variant, y_int_for_strategy, seed):
    minority = int(min(pd.Series(y_int_for_strategy).value_counts()))
    k = min(SMOTE_K_NEIGHBORS, max(1, minority - 1))
    strat = smote_sampling_strategy(y_int_for_strategy, variant)
    return SMOTE(sampling_strategy=strat, random_state=seed, k_neighbors=k)


# ============================ SMOTE-weighted wrapper ========================
class SMOTEWeightedPipeline(BaseEstimator, ClassifierMixin):
    """SMOTE -> RobustScaler -> classifier wrapper that PROPAGATES
    sample_weight through SMOTE.

    GridSearchCV's `gs.fit(X, y, sample_weight=sw)` calls our `fit(X, y,
    sample_weight=sw)` after subsetting both X/y and sw to the inner-train
    indices. Inside fit:
        1. SMOTE generates synthetic samples on (X, y) -> (X_aug, y_aug).
        2. We extend sw across the new length:
             - real samples : retain their composite confounder weight
             - synthetic    : weight = 1.0 (uniform; no metadata available)
           and re-normalize to mean 1.
        3. RobustScaler fits on X_aug.
        4. clf fits on scaler-transformed X_aug with the extended weights
           (when supports_sw=True; else fits without weights).
    """

    def __init__(self, clf=None, smote_variant="balanced",
                 random_state=42, k_neighbors=5, supports_sw=True):
        self.clf            = clf
        self.smote_variant  = smote_variant
        self.random_state   = random_state
        self.k_neighbors    = k_neighbors
        self.supports_sw    = supports_sw

    def _smote_strategy(self, y):
        if self.smote_variant == "balanced":
            return "auto"
        if self.smote_variant == "imbalanced":
            counts = pd.Series(y).value_counts().to_dict()
            return {int(c): int(2 * n) for c, n in counts.items()}
        raise ValueError(f"Unknown SMOTE variant: {self.smote_variant}")

    def fit(self, X, y, sample_weight=None):
        n_orig   = X.shape[0]
        minority = int(min(pd.Series(y).value_counts()))
        k        = min(self.k_neighbors, max(1, minority - 1))
        sm = SMOTE(sampling_strategy=self._smote_strategy(y),
                   random_state=self.random_state, k_neighbors=k)
        X_aug, y_aug = sm.fit_resample(X, y)
        n_synth = X_aug.shape[0] - n_orig

        self.scaler_ = RobustScaler()
        X_aug_scaled = self.scaler_.fit_transform(X_aug)

        self.clf_ = clone(self.clf)
        self.fitted_with_weights_ = False
        if sample_weight is not None and self.supports_sw:
            ext = np.concatenate([
                np.asarray(sample_weight, dtype=float),
                np.ones(n_synth, dtype=float),
            ])
            ext = ext / ext.mean()
            try:
                self.clf_.fit(X_aug_scaled, y_aug, sample_weight=ext)
                self.fitted_with_weights_ = True
            except TypeError:
                self.clf_.fit(X_aug_scaled, y_aug)
        else:
            self.clf_.fit(X_aug_scaled, y_aug)

        self.classes_       = self.clf_.classes_
        self.n_smote_orig_  = n_orig
        self.n_smote_synth_ = n_synth
        return self

    def predict(self, X):
        return self.clf_.predict(self.scaler_.transform(X))

    def predict_proba(self, X):
        return self.clf_.predict_proba(self.scaler_.transform(X))

    def decision_function(self, X):
        if hasattr(self.clf_, "decision_function"):
            return self.clf_.decision_function(self.scaler_.transform(X))
        raise AttributeError("Inner clf has no decision_function")


# ============================ Classifiers ==================================
def get_classifiers_with_smote(seed: int, variant: str) -> Dict[str, Dict]:
    """Each entry's `pipe` is a SMOTEWeightedPipeline wrapping the base
    classifier. Hyperparameter keys use the standard `clf__` prefix."""
    return {
        "mSVM-RBF": {
            "pipe": SMOTEWeightedPipeline(
                clf=SVC(kernel="rbf", probability=True, random_state=seed),
                smote_variant=variant, random_state=seed,
                k_neighbors=SMOTE_K_NEIGHBORS, supports_sw=True),
            "grid": {"clf__C":     [0.1, 1, 10],
                     "clf__gamma": ["scale", 0.01, 0.1, 1.0]},
        },
        "mSVM-Linear": {
            "pipe": SMOTEWeightedPipeline(
                clf=SVC(kernel="linear", probability=True, random_state=seed),
                smote_variant=variant, random_state=seed,
                k_neighbors=SMOTE_K_NEIGHBORS, supports_sw=True),
            "grid": {"clf__C": [0.01, 0.1, 1, 10]},
        },
        "mSVM-RFE": {
            "pipe": SMOTEWeightedPipeline(
                clf=RFE(SVC(kernel="linear", probability=True,
                            random_state=seed)),
                smote_variant=variant, random_state=seed,
                k_neighbors=SMOTE_K_NEIGHBORS, supports_sw=True),
            "grid": {"clf__n_features_to_select": None,
                     "clf__estimator__C":         [0.1, 1, 10]},
        },
        "kNN": {
            "pipe": SMOTEWeightedPipeline(
                clf=KNeighborsClassifier(),
                smote_variant=variant, random_state=seed,
                k_neighbors=SMOTE_K_NEIGHBORS, supports_sw=False),
            "grid": {"clf__n_neighbors": [3, 5, 7, 11, 15],
                     "clf__weights":     ["uniform", "distance"]},
        },
        "GaussianNB": {
            "pipe": SMOTEWeightedPipeline(
                clf=GaussianNB(),
                smote_variant=variant, random_state=seed,
                k_neighbors=SMOTE_K_NEIGHBORS, supports_sw=True),
            "grid": {"clf__var_smoothing": np.logspace(-11, -7, 5)},
        },
        "ElasticNet": {
            "pipe": SMOTEWeightedPipeline(
                clf=LogisticRegression(penalty="elasticnet", solver="saga",
                                       max_iter=5000, tol=1e-3,
                                       random_state=seed),
                smote_variant=variant, random_state=seed,
                k_neighbors=SMOTE_K_NEIGHBORS, supports_sw=True),
            "grid": {"clf__C":        [0.01, 0.1, 1, 10],
                     "clf__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]},
        },
    }


def rfe_grid_for_panel(k_top):
    base = sorted({max(2, k_top // 4), max(3, k_top // 2), k_top - 1, k_top})
    return [b for b in base if 2 <= b <= k_top]


# ============================ Metrics =======================================
def evaluate_binary(y_true, y_pred, y_proba_pos):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1":          f1_score(y_true, y_pred, average="macro"),
        "auroc":             roc_auc_score(y_true, y_proba_pos),
        "pr_auc":            average_precision_score(y_true, y_proba_pos),
        "sensitivity":       sens,
        "specificity":       spec,
        "mcc":               matthews_corrcoef(y_true, y_pred),
        "brier":             brier_score_loss(y_true, y_proba_pos),
        "cohen_kappa":       cohen_kappa_score(y_true, y_pred),
    }


def selected_genes_from_estimator(name: str, gs_result, gene_names):
    """Extract selected genes from the SMOTEWeightedPipeline wrapper."""
    best = gs_result.best_estimator_
    if name == "ElasticNet":
        coef = best.clf_.coef_
        mask = (np.abs(coef) > 1e-10).any(axis=0)
        return [g for g, m in zip(gene_names, mask) if m]
    if name == "mSVM-RFE":
        return [g for g, m in zip(gene_names, best.clf_.support_) if m]
    return []


# ============================ Aggregation ===================================
def aggregate_metrics(df):
    metric_cols = ["balanced_accuracy", "macro_f1", "auroc", "pr_auc",
                   "sensitivity", "specificity", "mcc", "brier", "cohen_kappa"]
    rows = []
    for clf, sub in df.groupby("classifier"):
        row = {"classifier": clf, "n_folds": len(sub)}
        for m in metric_cols:
            row[f"{m}_mean"] = sub[m].mean()
            row[f"{m}_std"]  = sub[m].std()
        rows.append(row)
    return pd.DataFrame(rows)


def consensus_hyperparams(df):
    rows = []
    for clf, sub in df.groupby("classifier"):
        params_list = [json.loads(p) for p in sub["best_params"]]
        keys = sorted({k for p in params_list for k in p.keys()})
        row = {"classifier": clf}
        for k in keys:
            vals = [p.get(k) for p in params_list]
            counter = Counter([str(v) for v in vals])
            mode_str, mode_count = counter.most_common(1)[0]
            row[k] = mode_str
            row[f"{k}__support_pct"] = round(100 * mode_count / len(vals), 1)
        rows.append(row)
    return pd.DataFrame(rows)


def fold_inclusion_table(sel_records, top_genes, n_folds_total):
    df = pd.DataFrame(sel_records)
    out = {}
    if df.empty:
        return out
    for clf in df["classifier"].unique():
        sub = df[df["classifier"] == clf]
        n_seen = sub.groupby("gene").size()
        full = pd.Series(0, index=top_genes, dtype=int)
        full.update(n_seen)
        out[clf] = pd.DataFrame({
            "gene": full.index,
            "n_folds_included": full.values,
            "fold_inclusion_pct":
                np.round(100 * full.values / n_folds_total, 1),
        }).sort_values("n_folds_included", ascending=False)
    return out


# ============================ Plot helpers =================================
def plot_roc_per_classifier(fold_df, out_path, title_suffix):
    fig, ax = plt.subplots(figsize=(8, 6))
    for clf, sub in fold_df.groupby("classifier"):
        y_true_all = np.concatenate([np.asarray(t) for t in sub["y_true"]])
        y_proba_all = np.concatenate([np.asarray(p) for p in sub["y_proba_pos"]])
        fpr, tpr, _ = roc_curve(y_true_all, y_proba_all)
        auc = roc_auc_score(y_true_all, y_proba_all)
        ax.plot(fpr, tpr, label=f"{clf} (AUC={auc:.3f})", linewidth=1.7)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, linewidth=1)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC across 27 outer folds (concatenated) - {title_suffix}")
    ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(out_path, dpi=200); plt.close(fig)


def plot_pr_per_classifier(fold_df, out_path, title_suffix):
    fig, ax = plt.subplots(figsize=(8, 6))
    for clf, sub in fold_df.groupby("classifier"):
        y_true_all = np.concatenate([np.asarray(t) for t in sub["y_true"]])
        y_proba_all = np.concatenate([np.asarray(p) for p in sub["y_proba_pos"]])
        prec, rec, _ = precision_recall_curve(y_true_all, y_proba_all)
        ap = average_precision_score(y_true_all, y_proba_all)
        ax.plot(rec, prec, label=f"{clf} (AP={ap:.3f})", linewidth=1.7)
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"PR across 27 outer folds (concatenated) - {title_suffix}")
    ax.legend(loc="lower left", fontsize=9); ax.grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(out_path, dpi=200); plt.close(fig)


# ============================ Per-classifier CV ============================
def run_classifier_cv(name, cfg, X, y_int, gene_names, meta,
                      fold_records, sel_records, smote_count_records,
                      variant):
    print(f"\n--- {name} ---")
    supports_sw = cfg["pipe"].supports_sw

    for seed in SEEDS:
        skf_outer = StratifiedKFold(n_splits=N_OUTER, shuffle=True,
                                     random_state=seed)
        for fold_i, (tr_idx, te_idx) in enumerate(skf_outer.split(X, y_int)):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y_int[tr_idx], y_int[te_idx]

            sw_tr = composite_sample_weights(meta.iloc[tr_idx], y_tr)

            n_pre = pd.Series(y_tr).value_counts().to_dict()
            sm_target = smote_sampling_strategy(y_tr, variant)
            n_post = (n_pre.copy() if isinstance(sm_target, str)
                      else sm_target)
            if isinstance(sm_target, str):
                maj = max(n_pre.values())
                n_post = {c: maj for c in n_pre.keys()}
            smote_count_records.append({
                "classifier": name, "seed": seed, "fold": fold_i + 1,
                "variant":    variant,
                "n_pre_HC":   int(n_pre.get(0, 0)),
                "n_pre_PD":   int(n_pre.get(1, 0)),
                "n_post_HC":  int(n_post.get(0, 0)),
                "n_post_PD":  int(n_post.get(1, 0)),
                "sw_supported": supports_sw,
            })

            inner = StratifiedKFold(n_splits=N_INNER, shuffle=True,
                                     random_state=seed * 1000 + fold_i)
            gs = GridSearchCV(
                estimator=cfg["pipe"], param_grid=cfg["grid"],
                scoring="balanced_accuracy", cv=inner,
                n_jobs=N_JOBS, refit=True,
            )
            if supports_sw:
                gs.fit(X_tr, y_tr, sample_weight=sw_tr)
            else:
                gs.fit(X_tr, y_tr)

            y_pred = gs.predict(X_te)
            y_proba_pos = gs.predict_proba(X_te)[:, 1]
            metrics = evaluate_binary(y_te, y_pred, y_proba_pos)

            fold_records.append({
                "classifier":   name,
                "seed":         seed,
                "fold":         fold_i + 1,
                "best_params":  json.dumps(gs.best_params_),
                "sw_used":      bool(supports_sw),
                **metrics,
                "y_true":       y_te.tolist(),
                "y_proba_pos":  y_proba_pos.tolist(),
            })
            print(f"  seed={seed} fold={fold_i+1}  "
                  f"BA={metrics['balanced_accuracy']:.3f}  "
                  f"AUROC={metrics['auroc']:.3f}  "
                  f"PR-AUC={metrics['pr_auc']:.3f}  "
                  f"sw={'on' if supports_sw else 'off'}", flush=True)

            if name in ("ElasticNet", "mSVM-RFE"):
                sel = selected_genes_from_estimator(name, gs, gene_names)
                for g in sel:
                    sel_records.append({"classifier": name, "seed": seed,
                                        "fold": fold_i + 1, "gene": g})


# ============================ Per-(variant, panel) run ======================
def run_for_variant(variant: str, panel_size: int):
    out_dir = OUTPUT_BASE / f"stage4_HP_PDHC_top{panel_size}_smote_{variant}"
    out_dir.mkdir(parents=True, exist_ok=True)
    s3_dir = OUTPUT_BASE / f"stage3_FS_PDHC_smote_{variant}"
    consensus_path = s3_dir / "consensus_panel.csv"

    print("\n" + "=" * 72)
    print(f"VARIANT: SMOTE = {variant.upper()}  (top {panel_size})")
    print(f"Stage-3 source: {s3_dir.name}")
    print(f"Output dir    : {out_dir.name}")
    print("=" * 72)

    if not consensus_path.exists():
        raise FileNotFoundError(
            f"{consensus_path} not found. Run stage3_FS_PDHC_smote.py first.")

    # ---- Top-K consensus genes from matching SMOTE FS variant ----
    consensus = pd.read_csv(consensus_path, index_col=0)
    if "fold_inclusion" not in consensus.columns:
        consensus.columns = ["fold_inclusion"]
    top_genes = (consensus.sort_values("fold_inclusion", ascending=False)
                          .head(panel_size).index.tolist())
    if len(top_genes) < panel_size:
        print(f"  WARN: consensus has only {len(top_genes)} genes "
              f"(< requested {panel_size}); proceeding with available set.")
    print(f"Top-{len(top_genes)} consensus genes loaded from {variant} Stage 3")

    # ---- Subset & save post-feature CSVs ----
    train_df = load_features_x_samples(TRAIN_PATH, top_genes).round(4)
    test_df  = load_features_x_samples(TEST_PATH,  top_genes).round(4)
    train_df.to_csv(out_dir / "train_postfeature.csv")
    test_df.to_csv(out_dir / "test_postfeature.csv")
    print(f"Pre-SMOTE post-feature CSVs saved (train: {train_df.shape}, "
          f"test: {test_df.shape})")

    # ---- Class labels ----
    meta_train = load_meta(train_df.columns)
    y_train_str = meta_train[CONDITION_COL].values
    y_train_bin = to_binary_labels(y_train_str)
    y_train_int = encode_binary(y_train_bin)
    print(f"Class balance (PD=1, HC=0): {dict(Counter(y_train_bin))}")

    X_train = train_df.T.values.astype(np.float64)
    gene_names = train_df.index.tolist()

    # ---- Save FULL-train SMOTE-augmented post-feature CSV ----
    sm_full = make_smote(variant, y_train_int, seed=42)
    X_train_smote, y_train_smote = sm_full.fit_resample(X_train, y_train_int)
    n_orig = X_train.shape[0]
    n_synth = X_train_smote.shape[0] - n_orig
    synth_ids = [f"SYNTH_{variant}_{i:03d}" for i in range(n_synth)]
    sample_ids_smote = list(train_df.columns) + synth_ids
    train_smote_df = pd.DataFrame(
        X_train_smote.T,
        index=gene_names,
        columns=sample_ids_smote
    ).round(4)
    train_smote_df.to_csv(out_dir / "train_postfeature_smote.csv")
    smote_meta = pd.DataFrame({
        "sample_id": sample_ids_smote,
        "condition": np.where(y_train_smote == 1, POS_LABEL, NEG_LABEL),
        "synthetic": [False] * n_orig + [True] * n_synth,
    })
    smote_meta.to_csv(out_dir / "train_postfeature_smote_meta.csv", index=False)
    print(f"Full-train SMOTE applied: {n_orig} real + {n_synth} synthetic = "
          f"{X_train_smote.shape[0]} samples")
    print(f"Synthetic class breakdown: HC={(y_train_smote[n_orig:]==0).sum()}, "
          f"PD={(y_train_smote[n_orig:]==1).sum()}")

    # ---- Build classifiers ----
    classifiers = get_classifiers_with_smote(seed=42, variant=variant)
    classifiers["mSVM-RFE"]["grid"]["clf__n_features_to_select"] = \
        rfe_grid_for_panel(len(top_genes))
    print(f"RFE n_features_to_select grid: "
          f"{classifiers['mSVM-RFE']['grid']['clf__n_features_to_select']}")

    # ---- Nested CV with SMOTE inside inner folds ----
    fold_records, sel_records, smote_count_records = [], [], []
    for name, cfg in classifiers.items():
        run_classifier_cv(name, cfg, X_train, y_train_int, gene_names,
                          meta_train,
                          fold_records, sel_records, smote_count_records,
                          variant)

    fold_df = pd.DataFrame(fold_records)
    csv_cols = [c for c in fold_df.columns
                if c not in ("y_true", "y_proba_pos")]
    fold_df[csv_cols].to_csv(out_dir / "fold_results.csv", index=False)

    pd.DataFrame(smote_count_records).to_csv(
        out_dir / "smote_fold_summary.csv", index=False)

    agg = aggregate_metrics(fold_df)
    agg.to_csv(out_dir / "final_metrics.csv", index=False)
    print(f"\n[{variant}] Final metrics (mean across 27 folds):")
    print(agg.set_index("classifier")
          [["balanced_accuracy_mean", "macro_f1_mean",
            "auroc_mean", "pr_auc_mean"]].round(3).to_string())

    cons = consensus_hyperparams(fold_df)
    cons.to_csv(out_dir / "consensus_hyperparams.csv", index=False)

    plot_roc_per_classifier(fold_df, out_dir / "roc_curves.png",
                             f"top {len(top_genes)}, SMOTE {variant}")
    plot_pr_per_classifier(fold_df,  out_dir / "pr_curves.png",
                            f"top {len(top_genes)}, SMOTE {variant}")

    n_folds_total = N_OUTER * len(SEEDS)
    inc_tables = fold_inclusion_table(sel_records, top_genes, n_folds_total)
    if "ElasticNet" in inc_tables:
        inc_tables["ElasticNet"].to_csv(
            out_dir / "enet_fold_inclusion.csv", index=False)
    if "mSVM-RFE" in inc_tables:
        inc_tables["mSVM-RFE"].to_csv(
            out_dir / "rfe_fold_inclusion.csv", index=False)

    print(f"\n[{variant}] Outputs in: {out_dir}")
    return out_dir


def main():
    print("=" * 72)
    print("STAGE 4 (PD vs HC) SMOTE - Multi-panel nested CV")
    print(f"Panel sizes  : {PANEL_SIZES}")
    print(f"SMOTE variants: {SMOTE_VARIANTS}")
    print(f"Outer folds={N_OUTER} x seeds={SEEDS} = {N_OUTER * len(SEEDS)}; "
          f"inner={N_INNER}")
    print(f"Total runs   : {len(PANEL_SIZES)} panels x {len(SMOTE_VARIANTS)} "
          f"variants = {len(PANEL_SIZES) * len(SMOTE_VARIANTS)}")
    print("=" * 72)
    for panel_size in PANEL_SIZES:
        for variant in SMOTE_VARIANTS:
            run_for_variant(variant, panel_size)


if __name__ == "__main__":
    main()


STAGE 4 (PD vs HC) SMOTE - Multi-panel nested CV
Panel sizes  : [5, 7, 10, 16]
SMOTE variants: ['balanced', 'imbalanced']
Outer folds=9 x seeds=[42, 123, 256] = 27; inner=4
Total runs   : 4 panels x 2 variants = 8

VARIANT: SMOTE = BALANCED  (top 5)
Stage-3 source: stage3_FS_PDHC_smote_balanced
Output dir    : stage4_HP_PDHC_top5_smote_balanced
Top-5 consensus genes loaded from balanced Stage 3
Pre-SMOTE post-feature CSVs saved (train: (5, 108), test: (5, 47))
Class balance (PD=1, HC=0): {np.str_('HC'): 36, np.str_('PD'): 72}
Full-train SMOTE applied: 108 real + 36 synthetic = 144 samples
Synthetic class breakdown: HC=36, PD=0
RFE n_features_to_select grid: [2, 3, 4, 5]

--- mSVM-RBF ---
  seed=42 fold=1  BA=0.688  AUROC=0.562  PR-AUC=0.714  sw=on
  seed=42 fold=2  BA=0.688  AUROC=0.875  PR-AUC=0.947  sw=on
  seed=42 fold=3  BA=0.562  AUROC=0.500  PR-AUC=0.752  sw=on
  seed=42 fold=4  BA=0.375  AUROC=0.469  PR-AUC=0.726  sw=on
  seed=42 fold=5  BA=0.562  AUROC=0.656  PR-AUC=0.849  sw=o